<a href="https://colab.research.google.com/github/Narashima1808/ML-Text-Classification-with-Feature-Analysis/blob/main/interpretable_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install wordcloud --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from wordcloud import WordCloud
import warnings
warnings.filterwarnings("ignore")

# Sklearn - Data
from sklearn.datasets import fetch_20newsgroups

# Sklearn - Preprocessing & Feature Extraction
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

# Sklearn - Models
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# Sklearn - Evaluation
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score
)
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Display settings
pd.set_option("display.max_colwidth", 100)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.family"] = "DejaVu Sans"



In [ ]:


"""
We use the 20 Newsgroups dataset — a benchmark NLP dataset with ~18,000
newsgroup posts across 20 categories. We pick 6 categories for clarity.
This mirrors real-world multi-class classification scenarios.
"""

# Select 6 semantically diverse categories
CATEGORIES = [
    'sci.med',
    'sci.space',
    'comp.graphics',
    'rec.sport.hockey',
    'talk.politics.guns',
    'alt.atheism'
]

# Load train/test splits
print("📥 Loading 20 Newsgroups dataset...")
train_data = fetch_20newsgroups(
    subset='train',
    categories=CATEGORIES,
    remove=('headers', 'footers', 'quotes'),  # Remove metadata — forces model to learn content
    random_state=42
)
test_data = fetch_20newsgroups(
    subset='test',
    categories=CATEGORIES,
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)

X_train_raw, y_train = train_data.data, train_data.target
X_test_raw,  y_test  = test_data.data,  test_data.target
class_names = train_data.target_names

print(f"\n Dataset Summary")
print(f"{'─'*40}")
print(f"  Training samples : {len(X_train_raw)}")
print(f"  Test samples     : {len(X_test_raw)}")
print(f"  Classes          : {len(class_names)}")
print(f"\n  Class Names:")
for i, name in enumerate(class_names):
    print(f"    [{i}] {name}")

# ── Class Distribution Plot ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Class Distribution in Train and Test Sets", fontsize=14, fontweight='bold')

for ax, (data, split) in zip(axes, [(y_train, "Train"), (y_test, "Test")]):
    counts = pd.Series(data).value_counts().sort_index()
    bars = ax.barh([class_names[i] for i in counts.index], counts.values,
                   color=plt.cm.Set2(np.linspace(0, 1, len(class_names))))
    ax.set_xlabel("Number of Samples")
    ax.set_title(f"{split} Set Distribution")
    ax.bar_label(bars, padding=3)
    ax.set_xlim(0, max(counts.values) * 1.15)

plt.tight_layout()
plt.savefig("class_distribution.png", bbox_inches='tight')
plt.show()

# ── Sample document lengths ──────────────────────────────────
train_lengths = [len(doc.split()) for doc in X_train_raw]
print(f"\n Document Length Stats (word count):")
print(f"  Mean   : {np.mean(train_lengths):.1f}")
print(f"  Median : {np.median(train_lengths):.1f}")
print(f"  Std    : {np.std(train_lengths):.1f}")
print(f"  Min    : {min(train_lengths)}")
print(f"  Max    : {max(train_lengths)}")

# ── Sample document ─────────────────────────────────────────
print(f"\n Sample Document (Class: {class_names[y_train[0]]}):")
print("─" * 60)
print(X_train_raw[0][:600])
print("─" * 60)

In [ ]:


"""
Preprocessing cleans raw text before feature extraction.
Steps: lowercasing, punctuation removal, stop word filtering.

Note: For classical ML with TF-IDF, stemming/lemmatization can help
but sklearn's built-in tokenizer + stop words often suffices.
We implement a custom cleaner for transparency.
"""

import re
import string

def preprocess_text(text):
    """
    Clean raw text:
      1. Lowercase
      2. Remove URLs
      3. Remove email addresses
      4. Remove numbers
      5. Remove punctuation
      6. Collapse whitespace
    """
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)          # Remove URLs
    text = re.sub(r'\S+@\S+', '', text)                  # Remove emails
    text = re.sub(r'\d+', '', text)                       # Remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()             # Collapse spaces
    return text

# Apply preprocessing
print("🔧 Preprocessing text...")
X_train = [preprocess_text(doc) for doc in X_train_raw]
X_test  = [preprocess_text(doc) for doc in X_test_raw]

# Verify
print(f"\n Preprocessing complete.")
print(f"\n  Before: {X_train_raw[0][:200]!r}")
print(f"\n  After : {X_train[0][:200]!r}")

# ── Length comparison ────────────────────────────────────────
raw_len   = np.mean([len(d.split()) for d in X_train_raw])
clean_len = np.mean([len(d.split()) for d in X_train])
print(f"\n  Avg words before : {raw_len:.1f}")
print(f"  Avg words after  : {clean_len:.1f}")
print(f"  Reduction        : {(1 - clean_len/raw_len)*100:.1f}%")

In [ ]:


"""
Two classical representations:

1. Bag-of-Words (BoW): Count of each word per document.
   → Simple, ignores word order, treats all words equally.

2. TF-IDF (Term Frequency–Inverse Document Frequency):
   TF(t,d)  = count(t in d) / total_words(d)
   IDF(t)   = log(N / df(t))   where df(t) = docs containing t
   TF-IDF   = TF × IDF
   → Downweights common words, upweights rare informative words.

Both use unigrams + bigrams (ngram_range=(1,2)) for richer features.
max_features=50000 to control vocabulary size.
"""

VECTORIZER_PARAMS = dict(
    ngram_range=(1, 2),
    max_features=50_000,
    min_df=2,           # Ignore terms in <2 docs (reduce noise)
    max_df=0.95,        # Ignore terms in >95% docs (too common)
    stop_words='english'
)

# ── Bag-of-Words ─────────────────────────────────────────────
bow_vectorizer = CountVectorizer(**VECTORIZER_PARAMS)
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow  = bow_vectorizer.transform(X_test)

# ── TF-IDF ───────────────────────────────────────────────────
tfidf_vectorizer = TfidfVectorizer(**VECTORIZER_PARAMS, sublinear_tf=True)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf  = tfidf_vectorizer.transform(X_test)

print(" Feature Matrix Summary")
print(f"{'─'*45}")
print(f"  BoW   Train shape : {X_train_bow.shape}   (sparse)")
print(f"  BoW   Test  shape : {X_test_bow.shape}")
print(f"  TF-IDF Train shape: {X_train_tfidf.shape}")
print(f"  TF-IDF Test  shape: {X_test_tfidf.shape}")
print(f"\n  Vocabulary size   : {len(bow_vectorizer.vocabulary_):,}")
sparsity = 1 - (X_train_bow.nnz / (X_train_bow.shape[0] * X_train_bow.shape[1]))
print(f"  Matrix sparsity   : {sparsity:.4%}")

# ── TF-IDF score visualization for one document ──────────────
doc_idx = 5
tfidf_scores = X_train_tfidf[doc_idx].toarray().flatten()
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
top_indices = tfidf_scores.argsort()[::-1][:15]

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(feature_names[top_indices][::-1], tfidf_scores[top_indices][::-1],
        color='steelblue', edgecolor='white')
ax.set_xlabel("TF-IDF Score")
ax.set_title(f"Top 15 TF-IDF Features — Document from class: '{class_names[y_train[doc_idx]]}'",
             fontweight='bold')
plt.tight_layout()
plt.savefig("tfidf_sample.png", bbox_inches='tight')
plt.show()

In [ ]:


"""
Three classical classifiers:

1. Multinomial Naive Bayes:
   P(class|doc) ∝ P(class) × ∏ P(word|class)
   → Fast, strong baseline, assumes feature independence.
   → Works natively with count features (BoW).
   → alpha: Laplace smoothing to handle unseen words.

2. Logistic Regression:
   P(y|x) = sigmoid(wᵀx + b)
   → Learns feature weights directly; highly interpretable.
   → C: inverse regularization strength (higher C = less regularization).
   → Excellent on TF-IDF features.

3. Linear SVM (LinearSVC):
   Maximize margin: argmin ½‖w‖² s.t. yᵢ(wᵀxᵢ+b) ≥ 1
   → State-of-the-art for text classification.
   → C: controls margin vs misclassification trade-off.
   → Very efficient with sparse feature matrices.
"""

# ── Define models ────────────────────────────────────────────
models = {
    "Naive Bayes":           MultinomialNB(alpha=0.1),
    "Logistic Regression":   LogisticRegression(C=5.0, max_iter=1000,
                                                 solver='lbfgs',
                                                 multi_class='multinomial',
                                                 random_state=42),
    "Linear SVM":            LinearSVC(C=1.0, max_iter=2000, random_state=42)
}

# ── Feature sets ─────────────────────────────────────────────
feature_sets = {
    "BoW":    (X_train_bow,   X_test_bow),
    "TF-IDF": (X_train_tfidf, X_test_tfidf)
}

# ── Training & Evaluation ────────────────────────────────────
results = []

print(" Training models...\n")
print(f"{'Model':<25} {'Features':<10} {'Train Acc':>10} {'Test Acc':>10} {'Macro F1':>10}")
print("─" * 70)

trained_models = {}

for feat_name, (X_tr, X_te) in feature_sets.items():
    for model_name, model in models.items():

        # Skip NB with TF-IDF (NB needs non-negative counts; TF-IDF with
        # sublinear_tf is still non-negative here, but conceptually BoW is
        # more natural for NB. We include both for comparison.)

        model_copy = type(model)(**model.get_params())
        model_copy.fit(X_tr, y_train)

        y_pred_tr = model_copy.predict(X_tr)
        y_pred_te = model_copy.predict(X_te)

        train_acc = accuracy_score(y_train, y_pred_tr)
        test_acc  = accuracy_score(y_test, y_pred_te)
        macro_f1  = f1_score(y_test, y_pred_te, average='macro')

        results.append({
            "Model":      model_name,
            "Features":   feat_name,
            "Train Acc":  round(train_acc, 4),
            "Test Acc":   round(test_acc, 4),
            "Macro F1":   round(macro_f1, 4),
        })

        key = f"{model_name}_{feat_name}"
        trained_models[key] = {
            "model": model_copy,
            "y_pred": y_pred_te
        }

        print(f"{model_name:<25} {feat_name:<10} {train_acc:>10.4f} {test_acc:>10.4f} {macro_f1:>10.4f}")

results_df = pd.DataFrame(results)
print("\n Training complete!")

In [ ]:


"""
A single train/test split can be lucky or unlucky.
5-Fold Stratified Cross-Validation gives a reliable estimate
of generalization with mean ± std across folds.
"""

print("Running 5-Fold Stratified Cross-Validation...\n")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []

# CV only on TF-IDF (best representation per initial results)
for model_name, model in models.items():
    scores = cross_val_score(model, X_train_tfidf, y_train,
                             cv=cv, scoring='f1_macro', n_jobs=-1)
    cv_results.append({
        "Model": model_name,
        "CV Mean F1": round(scores.mean(), 4),
        "CV Std":     round(scores.std(), 4),
        "Scores":     scores
    })
    print(f"  {model_name:<25}: {scores.mean():.4f} ± {scores.std():.4f}")

# ── Plot CV results ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#4CAF50', '#2196F3', '#FF5722']

for i, res in enumerate(cv_results):
    x = np.arange(5) + i * 0.25
    ax.bar(x, res["Scores"], width=0.22, label=res["Model"],
           color=colors[i], alpha=0.85, edgecolor='white')

ax.axhline(y=np.mean([r["CV Mean F1"] for r in cv_results]),
           color='red', linestyle='--', linewidth=1, label='Overall Mean')
ax.set_xlabel("CV Fold")
ax.set_ylabel("Macro F1 Score")
ax.set_title("5-Fold Cross-Validation — Macro F1 per Model (TF-IDF)", fontweight='bold')
ax.set_xticks(np.arange(5) + 0.25)
ax.set_xticklabels([f"Fold {i+1}" for i in range(5)])
ax.legend()
ax.set_ylim(0.7, 1.0)
plt.tight_layout()
plt.savefig("cross_validation.png", bbox_inches='tight')
plt.show()

In [ ]:




print(" Full Results Table")
print("=" * 70)
display(results_df.sort_values("Test Acc", ascending=False).reset_index(drop=True))

# ── Grouped Bar Chart ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Model Comparison: BoW vs TF-IDF", fontsize=14, fontweight='bold')

metrics = ["Test Acc", "Macro F1"]
titles  = ["Test Accuracy", "Macro F1 Score"]

for ax, metric, title in zip(axes, metrics, titles):
    bow_vals   = results_df[results_df.Features=="BoW"][metric].values
    tfidf_vals = results_df[results_df.Features=="TF-IDF"][metric].values
    model_names = results_df[results_df.Features=="BoW"]["Model"].values

    x = np.arange(len(model_names))
    w = 0.35

    b1 = ax.bar(x - w/2, bow_vals,   w, label='BoW',    color='#5C85D6', alpha=0.9)
    b2 = ax.bar(x + w/2, tfidf_vals, w, label='TF-IDF', color='#E05C5C', alpha=0.9)

    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=10)
    ax.set_ylabel(metric)
    ax.set_title(title, fontweight='bold')
    ax.legend()
    ax.set_ylim(0.7, 1.0)
    ax.bar_label(b1, fmt='%.3f', fontsize=8, padding=2)
    ax.bar_label(b2, fmt='%.3f', fontsize=8, padding=2)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("model_comparison.png", bbox_inches='tight')
plt.show()

# Best model
best_row = results_df.sort_values("Test Acc", ascending=False).iloc[0]
print(f"\n Best Model : {best_row['Model']} with {best_row['Features']}")
print(f"   Test Accuracy : {best_row['Test Acc']}")
print(f"   Macro F1      : {best_row['Macro F1']}")

In [ ]:

#  Confusion Matrix Analysis


"""
The confusion matrix shows which classes are misclassified as which.
Off-diagonal entries = errors. Darker diagonal = better model.
"""

# Use best model: SVM + TF-IDF
best_key = "Linear SVM_TF-IDF"
y_pred_best = trained_models[best_key]["y_pred"]

cm = confusion_matrix(y_test, y_pred_best)
cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True)

short_names = [n.split('.')[-1] for n in class_names]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Confusion Matrix — Linear SVM + TF-IDF", fontsize=13, fontweight='bold')

for ax, data, title, fmt in zip(
    axes,
    [cm, cm_normalized],
    ["Raw Counts", "Normalized (Row %)"],
    ['d', '.2f']
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=short_names, yticklabels=short_names,
                linewidths=0.5, linecolor='white', ax=ax,
                cbar_kws={'shrink': 0.8})
    ax.set_xlabel("Predicted Label", fontsize=11)
    ax.set_ylabel("True Label", fontsize=11)
    ax.set_title(title, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig("confusion_matrix.png", bbox_inches='tight')
plt.show()

# ── Per-class breakdown ──────────────────────────────────────
print("\n Per-Class Classification Report (Linear SVM + TF-IDF):")
print("─" * 60)
print(classification_report(y_test, y_pred_best, target_names=class_names))

In [ ]:




"""
INTERPRETABILITY — the core ask of the project.

For Logistic Regression: coefficients directly represent
  feature importance per class.
  Positive coef → word pushes toward that class.

For Naive Bayes: log P(word|class) - log P(word|other)
  gives discriminative words per class.

For SVM (LinearSVC): decision function coefficients behave
  similarly to logistic regression weights.

We visualize top-N words per class for each model.
"""

# ── Helper ───────────────────────────────────────────────────
def plot_top_features(model, vectorizer, class_names, n=12, title=""):
    feature_names = np.array(vectorizer.get_feature_names_out())
    n_classes = len(class_names)
    ncols = 3
    nrows = (n_classes + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 3.5))
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.01)
    axes = axes.flatten()

    if hasattr(model, 'coef_'):
        coef_matrix = model.coef_    # shape: (n_classes, n_features)
    else:
        # NB: use feature_log_prob_
        coef_matrix = model.feature_log_prob_

    for i, (ax, cls) in enumerate(zip(axes, class_names)):
        coefs = coef_matrix[i]
        top_pos = np.argsort(coefs)[-n:][::-1]

        ax.barh(feature_names[top_pos][::-1], coefs[top_pos][::-1],
                color=plt.cm.tab10(i / n_classes), edgecolor='white')
        ax.set_title(f"Class: {cls.split('.')[-1]}", fontsize=10, fontweight='bold')
        ax.set_xlabel("Weight / Log-Prob")
        ax.tick_params(labelsize=8)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    return fig


# ── Logistic Regression feature weights ──────────────────────
lr_model   = trained_models["Logistic Regression_TF-IDF"]["model"]
fig = plot_top_features(lr_model, tfidf_vectorizer, class_names,
                        title="Top Discriminative Features — Logistic Regression (TF-IDF)")
plt.savefig("feature_importance_lr.png", bbox_inches='tight')
plt.show()

# ── Naive Bayes (log prob) ────────────────────────────────────
nb_model   = trained_models["Naive Bayes_BoW"]["model"]
fig = plot_top_features(nb_model, bow_vectorizer, class_names,
                        title="Top Features — Naive Bayes (BoW log-probabilities)")
plt.savefig("feature_importance_nb.png", bbox_inches='tight')
plt.show()

# ── Linear SVM weights ────────────────────────────────────────
svm_model  = trained_models["Linear SVM_TF-IDF"]["model"]
fig = plot_top_features(svm_model, tfidf_vectorizer, class_names,
                        title="Top Discriminative Features — Linear SVM (TF-IDF)")
plt.savefig("feature_importance_svm.png", bbox_inches='tight')
plt.show()

In [ ]:




"""
Word clouds visualize the most important vocabulary per class
using TF-IDF weights from the Logistic Regression model.
Larger word = higher importance to that class.
"""

feature_names = tfidf_vectorizer.get_feature_names_out()
coef_matrix   = lr_model.coef_

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Word Clouds — Most Discriminative Words per Class (LR Weights)",
             fontsize=14, fontweight='bold')

cmaps = ['Blues', 'Greens', 'Reds', 'Purples', 'Oranges', 'YlOrBr']

for i, (ax, cls) in enumerate(zip(axes.flatten(), class_names)):
    coefs = coef_matrix[i]
    # Only keep positive weights
    pos_mask = coefs > 0
    word_weights = dict(zip(feature_names[pos_mask], coefs[pos_mask]))

    wc = WordCloud(
        width=400, height=300,
        background_color='white',
        colormap=cmaps[i],
        max_words=60,
        prefer_horizontal=0.9
    ).generate_from_frequencies(word_weights)

    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f"{cls.split('.')[-1]}", fontsize=11, fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig("word_clouds.png", bbox_inches='tight')
plt.show()

In [ ]:




"""
Error analysis helps understand WHERE and WHY a model fails.
Steps:
  1. Identify misclassified samples
  2. Examine confusion patterns (which class → which class)
  3. Look at confidence of wrong predictions
  4. Find most confidently wrong predictions (worst errors)
"""

# Use LR for error analysis (gives probabilities)
lr_tfidf_model = trained_models["Logistic Regression_TF-IDF"]["model"]
y_pred_lr      = trained_models["Logistic Regression_TF-IDF"]["y_pred"]

# Probabilities for confidence analysis
probs = lr_tfidf_model.predict_proba(X_test_tfidf)

# ── Build error DataFrame ────────────────────────────────────
error_df = pd.DataFrame({
    "text":        [doc[:300] for doc in X_test],
    "true_label":  [class_names[y] for y in y_test],
    "pred_label":  [class_names[p] for p in y_pred_lr],
    "confidence":  probs.max(axis=1),
    "correct":     y_test == y_pred_lr
})

errors = error_df[~error_df.correct].copy()
correct = error_df[error_df.correct].copy()

print(f" Error Analysis Summary")
print(f"{'─'*40}")
print(f"  Total test samples  : {len(error_df)}")
print(f"  Correctly classified: {len(correct)} ({len(correct)/len(error_df)*100:.1f}%)")
print(f"  Misclassified       : {len(errors)} ({len(errors)/len(error_df)*100:.1f}%)")

# ── Confidence distribution: correct vs wrong ────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(correct.confidence, bins=30, color='#4CAF50', alpha=0.7, label='Correct', edgecolor='white')
axes[0].hist(errors.confidence,  bins=30, color='#F44336', alpha=0.7, label='Errors',  edgecolor='white')
axes[0].set_xlabel("Prediction Confidence")
axes[0].set_ylabel("Count")
axes[0].set_title("Confidence Distribution: Correct vs Misclassified", fontweight='bold')
axes[0].legend()
axes[0].axvline(errors.confidence.mean(), color='red', linestyle='--',
                label=f"Error mean: {errors.confidence.mean():.2f}")

# ── Error heatmap: true → predicted ─────────────────────────
error_cm = pd.crosstab(
    errors.true_label.apply(lambda x: x.split('.')[-1]),
    errors.pred_label.apply(lambda x: x.split('.')[-1]),
    rownames=["True"], colnames=["Predicted"]
)
sns.heatmap(error_cm, annot=True, fmt='d', cmap='OrRd',
            linewidths=0.5, linecolor='white', ax=axes[1],
            cbar_kws={'shrink': 0.8})
axes[1].set_title("Error Confusion Heatmap (Misclassified Only)", fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig("error_analysis.png", bbox_inches='tight')
plt.show()

# ── Most confidently wrong predictions ──────────────────────
print(f"\n Top 5 Most Confidently Wrong Predictions:")
print("=" * 80)
worst = errors.sort_values("confidence", ascending=False).head(5)
for _, row in worst.iterrows():
    print(f"\n  True     : {row.true_label}")
    print(f"  Predicted: {row.pred_label}  (confidence: {row.confidence:.3f})")
    print(f"  Text     : {row.text[:250]}...")
    print("─" * 80)

In [ ]:


"""
Let's concretely show HOW BoW and TF-IDF differ on the same document.
"""

sample_idx = 20
doc = X_train[sample_idx]
label = class_names[y_train[sample_idx]]

# Get BoW and TF-IDF vectors
bow_vec   = bow_vectorizer.transform([doc])
tfidf_vec = tfidf_vectorizer.transform([doc])

bow_words   = np.array(bow_vectorizer.get_feature_names_out())
tfidf_words = np.array(tfidf_vectorizer.get_feature_names_out())

# Top words by each representation
bow_scores   = bow_vec.toarray().flatten()
tfidf_scores = tfidf_vec.toarray().flatten()

top_bow   = np.argsort(bow_scores)[::-1][:15]
top_tfidf = np.argsort(tfidf_scores)[::-1][:15]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(f"BoW vs TF-IDF — Sample Document from '{label.split('.')[-1]}'",
             fontsize=13, fontweight='bold')

axes[0].barh(bow_words[top_bow][::-1], bow_scores[top_bow][::-1],
             color='#5C85D6', edgecolor='white')
axes[0].set_title("Bag-of-Words (Raw Counts)", fontweight='bold')
axes[0].set_xlabel("Word Count")

axes[1].barh(tfidf_words[top_tfidf][::-1], tfidf_scores[top_tfidf][::-1],
             color='#E05C5C', edgecolor='white')
axes[1].set_title("TF-IDF Scores", fontweight='bold')
axes[1].set_xlabel("TF-IDF Score")

plt.tight_layout()
plt.savefig("bow_vs_tfidf.png", bbox_inches='tight')
plt.show()

print(f"\n Key Insight:")
print(f"  BoW highlights frequent words (often stop words remain after vectorizer).")
print(f"  TF-IDF suppresses common corpus-wide terms and lifts domain-specific rare words.")
print(f"  This is why TF-IDF consistently outperforms BoW in text classification.")

In [ ]:


fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor('#F8F9FA')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── 1. Model comparison bar ──────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
pivot = results_df.pivot(index='Model', columns='Features', values='Test Acc')
pivot.plot(kind='bar', ax=ax1, color=['#5C85D6', '#E05C5C'],
           edgecolor='white', alpha=0.9)
ax1.set_title("Test Accuracy: All Models × Feature Sets", fontweight='bold')
ax1.set_ylabel("Test Accuracy")
ax1.set_ylim(0.75, 1.00)
ax1.tick_params(axis='x', rotation=15)
ax1.legend(title="Features")
ax1.grid(axis='y', alpha=0.3)
for bar in ax1.patches:
    ax1.annotate(f'{bar.get_height():.3f}',
                 (bar.get_x() + bar.get_width()/2, bar.get_height()),
                 ha='center', va='bottom', fontsize=8)

# ── 2. Summary table ─────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
ax2.axis('off')
best = results_df.sort_values("Test Acc", ascending=False)
table_data = [["Model", "Feat", "Acc", "F1"]] + \
             [[r.Model.split()[-1], r.Features,
               f"{r['Test Acc']:.3f}", f"{r['Macro F1']:.3f}"]
              for _, r in best.iterrows()]
tbl = ax2.table(cellText=table_data[1:], colLabels=table_data[0],
                cellLoc='center', loc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.2, 1.5)
ax2.set_title("Results Table", fontweight='bold', pad=20)

# ── 3. CV scores ─────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
cv_means = [r["CV Mean F1"] for r in cv_results]
cv_stds  = [r["CV Std"] for r in cv_results]
model_labels = [r["Model"].split()[-1] for r in cv_results]
bars = ax3.bar(model_labels, cv_means, yerr=cv_stds, capsize=5,
               color=['#4CAF50', '#2196F3', '#FF5722'], alpha=0.85)
ax3.set_title("5-Fold CV Macro F1 (TF-IDF)", fontweight='bold')
ax3.set_ylabel("Macro F1")
ax3.set_ylim(0.8, 1.0)
ax3.grid(axis='y', alpha=0.3)

# ── 4. Error distribution ────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
err_by_class = errors.groupby('true_label').size()
err_by_class.index = err_by_class.index.map(lambda x: x.split('.')[-1])
ax4.bar(err_by_class.index, err_by_class.values,
        color=plt.cm.Set3(np.linspace(0, 1, len(err_by_class))), edgecolor='white')
ax4.set_title("Errors per True Class (LR)", fontweight='bold')
ax4.set_ylabel("# Errors")
ax4.tick_params(axis='x', rotation=30)
ax4.grid(axis='y', alpha=0.3)

# ── 5. Feature count impact ──────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
max_feats_list = [1000, 5000, 10000, 30000, 50000]
f1_scores_feats = []
for mf in max_feats_list:
    v = TfidfVectorizer(max_features=mf, ngram_range=(1,2),
                        stop_words='english', sublinear_tf=True)
    Xtr = v.fit_transform(X_train)
    Xte = v.transform(X_test)
    m = LinearSVC(C=1.0, max_iter=2000, random_state=42)
    m.fit(Xtr, y_train)
    f1_scores_feats.append(f1_score(y_test, m.predict(Xte), average='macro'))

ax5.plot(max_feats_list, f1_scores_feats, 'o-', color='purple',
         linewidth=2, markersize=7)
ax5.set_title("Macro F1 vs Vocabulary Size (SVM+TF-IDF)", fontweight='bold')
ax5.set_xlabel("Max Features")
ax5.set_ylabel("Macro F1")
ax5.set_xscale('log')
ax5.grid(True, alpha=0.3)

plt.suptitle(" Final Summary Dashboard — Text Classification Project",
             fontsize=15, fontweight='bold', y=1.02)
plt.savefig("final_dashboard.png", bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()